# `fact_sale` Row-Group Analysis and Remediation

These are potential considerations based on analyzed metadata rather than hard best practices. The ideal choice depends on your data volumes and query patterns.

**Reported observation:** the largest `fact_sale` row group has **25,610,000 rows**, above the approximate **16,000,000-row** consideration threshold. Very large row groups can potentially reduce scan parallelism; rewriting or optimizing toward roughly **8–16 million rows** may help.

This notebook dynamically inspects the current Delta/Parquet layout, estimates a rewrite partition count targeting approximately **8–16 million rows per output partition**, and validates row-group metadata before and after optional remediation.

> Data-changing actions are disabled by default. Review the generated table names and flags before enabling a rewrite or `OPTIMIZE`.

## 1. Configure the Microsoft Fabric Spark Session

Attach the Lakehouse containing `fact_sale` before running the notebook. Adjust the parameters below for schema-qualified tables or a different target. The 12 million target is a planning estimate within the 8–16 million guidance band; Parquet row-group sizes are also influenced by schema width, compression, runtime, and write settings.

In [ ]:
import math
import time
from urllib.parse import unquote, urlparse

import pandas as pd
import pyarrow.parquet as pq

TABLE_NAME = "fact_sale"
REWRITE_TABLE_NAME = "fact_sale_rowgroup_rewrite"
TARGET_ROWS_PER_PARTITION = 12_000_000
TARGET_MIN_ROWS = 8_000_000
TARGET_MAX_ROWS = 16_000_000
WARNING_THRESHOLD_ROWS = 16_000_000
APPLY_REWRITE = False
APPLY_OPTIMIZE = False

# Optional Fabric write settings. These influence file layout but do not guarantee
# a specific number of rows in each Parquet row group.
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")


def quote_table_name(table_name):
    return ".".join(f"`{part.replace('`', '``')}`" for part in table_name.split("."))


TABLE_SQL = quote_table_name(TABLE_NAME)
REWRITE_TABLE_SQL = quote_table_name(REWRITE_TABLE_NAME)
print(f"Source table: {TABLE_NAME}")
print(f"Rewrite table: {REWRITE_TABLE_NAME}")
print(f"Data-changing actions enabled: rewrite={APPLY_REWRITE}, optimize={APPLY_OPTIMIZE}")

## 2. Inspect the `fact_sale` Delta Table

This section verifies that the attached Lakehouse contains the table, captures table detail and partition information, counts rows, and records a baseline `COUNT(*)` duration. It then reads metadata for active Parquet files only, without scanning their data pages.

In [ ]:
if not spark.catalog.tableExists(TABLE_NAME):
    raise ValueError(
        f"Table {TABLE_NAME!r} was not found. Attach the correct default Lakehouse "
        "or set TABLE_NAME to a schema-qualified table name."
    )

detail = spark.sql(f"DESCRIBE DETAIL {TABLE_SQL}").first().asDict(recursive=True)
if str(detail.get("format", "")).lower() != "delta":
    raise ValueError(f"{TABLE_NAME!r} is not a Delta table: format={detail.get('format')!r}")

table_location = detail["location"]
partition_columns = detail.get("partitionColumns") or []

baseline_started = time.perf_counter()
total_rows = spark.sql(f"SELECT COUNT(*) AS row_count FROM {TABLE_SQL}").first()["row_count"]
baseline_count_seconds = time.perf_counter() - baseline_started

print(f"Location: {table_location}")
print(f"Partition columns: {partition_columns or 'none'}")
print(f"Rows: {total_rows:,}")
print(f"Files reported by DESCRIBE DETAIL: {detail.get('numFiles', 'n/a')}")
print(f"Table size: {detail.get('sizeInBytes', 0) / (1024 ** 3):,.2f} GiB")
print(f"Baseline COUNT(*): {baseline_count_seconds:,.2f} seconds")

In [ ]:
def to_fabric_local_path(file_uri):
    decoded = unquote(file_uri)
    if decoded.startswith("file:"):
        return urlparse(decoded).path
    if "/Tables/" in decoded:
        relative_table_path = decoded.split("/Tables/", 1)[1]
        return f"/lakehouse/default/Tables/{relative_table_path}"
    if decoded.startswith("/"):
        return decoded
    raise ValueError(f"Cannot map file URI to the attached Lakehouse mount: {file_uri}")


def inspect_row_groups(table_name):
    active_files = spark.table(table_name).inputFiles()
    if not active_files:
        raise ValueError(f"No active Parquet files were found for {table_name!r}.")

    records = []
    failures = []
    for file_uri in active_files:
        try:
            local_path = to_fabric_local_path(file_uri)
            parquet_metadata = pq.ParquetFile(local_path).metadata
            for row_group_index in range(parquet_metadata.num_row_groups):
                row_group = parquet_metadata.row_group(row_group_index)
                records.append(
                    {
                        "file_path": file_uri,
                        "row_group_index": row_group_index,
                        "rows": row_group.num_rows,
                        "size_bytes": row_group.total_byte_size,
                    }
                )
        except Exception as exc:
            failures.append((file_uri, str(exc)))

    if not records:
        example = failures[0] if failures else ("n/a", "unknown error")
        raise RuntimeError(
            "Parquet metadata could not be read through /lakehouse/default. "
            "Attach the source Lakehouse as the notebook's default Lakehouse and retry. "
            f"First failure: {example[0]}: {example[1]}"
        )
    if failures:
        print(f"Warning: skipped {len(failures):,} of {len(active_files):,} active files.")

    result = pd.DataFrame.from_records(records)
    result["size_mib"] = result["size_bytes"] / (1024 ** 2)
    return result.sort_values("rows", ascending=False, ignore_index=True)


baseline_row_groups = inspect_row_groups(TABLE_NAME)
display(baseline_row_groups.head(20))

In [ ]:
def assess_row_groups(row_groups, table_name):
    largest = int(row_groups["rows"].max())
    oversized_count = int((row_groups["rows"] > WARNING_THRESHOLD_ROWS).sum())
    within_target_count = int(
        row_groups["rows"].between(TARGET_MIN_ROWS, TARGET_MAX_ROWS, inclusive="both").sum()
    )

    print(f"Table: {table_name}")
    print(f"Row groups inspected: {len(row_groups):,}")
    print(f"Largest row group: {largest:,} rows")
    print(f"Row groups above ~{WARNING_THRESHOLD_ROWS:,}: {oversized_count:,}")
    print(f"Row groups in the {TARGET_MIN_ROWS:,}–{TARGET_MAX_ROWS:,} range: {within_target_count:,}")

    if oversized_count:
        print("\nPotential consideration — Row groups may be too large")
        print(
            f"The largest row group has {largest:,} rows, above ~{WARNING_THRESHOLD_ROWS:,} rows. "
            "Very large row groups can potentially have a negative impact on performance and "
            "reduce scan parallelism. Rewriting or running OPTIMIZE so row groups stay around "
            f"{TARGET_MIN_ROWS // 1_000_000}–{TARGET_MAX_ROWS // 1_000_000} million rows may help."
        )
        print(
            "This is guidance based on analyzed metadata rather than a hard best practice; "
            "the ideal choice depends on data volumes and query patterns."
        )
    else:
        print("\nNo row group exceeds the configured warning threshold.")

    return {
        "largest_rows": largest,
        "oversized_count": oversized_count,
        "within_target_count": within_target_count,
    }


baseline_assessment = assess_row_groups(baseline_row_groups, TABLE_NAME)

## 3. Calculate the Rewrite Partition Count

The estimate below uses total rows divided by the configurable 12 million row target. It is a starting point, not a guarantee: Spark may produce multiple Parquet row groups per file, and skew or partition columns can change the resulting distribution.

In [ ]:
rewrite_partition_count = max(1, math.ceil(total_rows / TARGET_ROWS_PER_PARTITION))
estimated_rows_per_partition = math.ceil(total_rows / rewrite_partition_count)

print(f"Estimated output partitions: {rewrite_partition_count:,}")
print(f"Estimated rows per output partition: {estimated_rows_per_partition:,}")
print(
    "Estimate is within target band: "
    f"{TARGET_MIN_ROWS <= estimated_rows_per_partition <= TARGET_MAX_ROWS}"
)

## 4. Rewrite and Optimize the Table

The rewrite is deliberately directed to `REWRITE_TABLE_NAME`, leaving `fact_sale` unchanged. Enable `APPLY_REWRITE` only after reviewing capacity, storage headroom, partitioning, concurrent workloads, and downstream dependencies. `OPTIMIZE` is independently guarded and runs only against the rewrite table.

`OPTIMIZE` can improve file layout, but it does **not** guarantee a particular number of rows per Parquet row group. Validate the actual metadata afterward.

In [ ]:
if APPLY_REWRITE:
    source_df = spark.table(TABLE_NAME).repartition(rewrite_partition_count)
    writer = (
        source_df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
    )
    if partition_columns:
        writer = writer.partitionBy(*partition_columns)
    writer.saveAsTable(REWRITE_TABLE_NAME)
    print(f"Rewrote {TABLE_NAME} to separate table {REWRITE_TABLE_NAME}.")
else:
    print("Rewrite skipped. Set APPLY_REWRITE = True after reviewing the plan.")

if APPLY_OPTIMIZE:
    if not spark.catalog.tableExists(REWRITE_TABLE_NAME):
        raise ValueError(
            f"Cannot optimize {REWRITE_TABLE_NAME!r}: the rewrite table does not exist. "
            "Run the guarded rewrite first or change REWRITE_TABLE_NAME explicitly."
        )
    spark.sql(f"OPTIMIZE {REWRITE_TABLE_SQL}")
    print(f"OPTIMIZE completed for {REWRITE_TABLE_NAME}.")
else:
    print("OPTIMIZE skipped. Set APPLY_OPTIMIZE = True to optimize the rewrite table.")

## 5. Validate Row Groups and Query Performance

When the rewrite table exists, this section checks row counts, reruns the Parquet metadata assessment, and compares `COUNT(*)` duration with the baseline. Query timing is directional only; cache state and concurrent Fabric capacity activity can materially affect results.

In [ ]:
if spark.catalog.tableExists(REWRITE_TABLE_NAME):
    rewritten_started = time.perf_counter()
    rewritten_rows = spark.sql(
        f"SELECT COUNT(*) AS row_count FROM {REWRITE_TABLE_SQL}"
    ).first()["row_count"]
    rewritten_count_seconds = time.perf_counter() - rewritten_started

    if rewritten_rows != total_rows:
        raise AssertionError(
            f"Row-count mismatch: source={total_rows:,}, rewrite={rewritten_rows:,}"
        )

    rewritten_row_groups = inspect_row_groups(REWRITE_TABLE_NAME)
    rewritten_assessment = assess_row_groups(rewritten_row_groups, REWRITE_TABLE_NAME)

    comparison = pd.DataFrame(
        [
            {
                "table": TABLE_NAME,
                "rows": total_rows,
                "largest_row_group_rows": baseline_assessment["largest_rows"],
                "oversized_row_groups": baseline_assessment["oversized_count"],
                "count_seconds": baseline_count_seconds,
            },
            {
                "table": REWRITE_TABLE_NAME,
                "rows": rewritten_rows,
                "largest_row_group_rows": rewritten_assessment["largest_rows"],
                "oversized_row_groups": rewritten_assessment["oversized_count"],
                "count_seconds": rewritten_count_seconds,
            },
        ]
    )
    display(comparison)
else:
    print(
        f"Validation skipped because {REWRITE_TABLE_NAME!r} does not exist. "
        "Run the guarded rewrite, then rerun this cell."
    )